In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html#sklearn.ensemble.HistGradientBoostingRegressor

# df = pd.read_csv('/Users/nrcase/CSC522/CSC522-Project/dataset_with_labels.csv')
df = pd.read_csv('../dataset_with_labels.csv')
df.head()

In [ ]:

from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    HistGradientBoostingRegressor()
)

# Tuning Hyperparameters

In [ ]:
def explore_single_hp_values(pipeline, param_name, param_values, X_train, y_train, k_fold, scoring):
    param_grid = {param_name: param_values}
    grid_search = GridSearchCV(pipeline, param_grid, cv=k_fold, scoring=scoring)
    grid_search.fit(X_train, y_train)
    result_columns = [f"param_{param_name}", "mean_test_score", "std_test_score", "rank_test_score"]
    return pd.DataFrame(grid_search.cv_results_)[result_columns]
  
import seaborn as sns
import matplotlib.pyplot as plt

def plot_gridsearch_heatmap(results_df, x_param, y_param, score='neg_mean_squared_error'):    
    # Pivot the table to format it for a heatmap
    heatmap_data = results_df.pivot(index=f'param_{y_param}', columns=f'param_{x_param}', values=score)
    
    # Plot heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(heatmap_data, annot=True, cmap='viridis', fmt='.3f', linewidths=0.5)
    plt.title(f'Grid Search Results: {score}')
    plt.xlabel(x_param)
    plt.ylabel(y_param)
    plt.show()

In [ ]:
# Hyperparameter Tuning
hgbr_step_name = pipeline.steps[1][0]
step_prefix = '__'
hgbr_step_prefix = f"{hgbr_step_name}{step_prefix}"

# Parameter Keys
loss_key = hgbr_step_prefix + "loss"
max_leaf_key = hgbr_step_prefix + "max_leaf_nodes"
max_depth_key = hgbr_step_prefix + "max_depth"

# hgbr_param_keys = HistGradientBoostingRegressor().get_params().keys()
# for key in [loss_key, learning_rate_key, max_leaf_key, max_depth_key]:
#     assert key.startswith(hgbr_step_name), f"Key {key} should start with {hgbr_step_name}"
#     assert "__" in key, f"Key {key} should be connected by a double underscope __"
#     assert key[key.index("__") + 2:] in hgbr_param_keys, f"Key {key} should end with a DT parameter name"

from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

k_fold = KFold(n_splits=5, shuffle=True, random_state=42)

# Define the param_grid and grid_search
param_grid = {
  loss_key: ['squared_error', 'absolute_error'],
  max_leaf_key: [20, 25, 30, 35, 40],
  max_depth_key:[5,15,25,35]
}
grid_search = GridSearchCV(pipeline, param_grid, scoring="neg_mean_squared_error", cv=k_fold)
grid_search.fit(X_train, y_train)

# Print the best parameters
grid_search.best_params_
grid_search.best_score_

In [ ]:
# results = pd.DataFrame(grid_search.cv_results_)
# results = results[results['param_' + loss_key] == 'squared_error']
# plot_gridsearch_heatmap(results, max_leaf_key, max_depth_key)
grid_search.best_params_

In [ ]:
print("Pre-tuned")
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))

# Fitting Pipeline w/ Tuned HPs

In [ ]:
pipeline_tuned = make_pipeline(
    preprocessing,
    HistGradientBoostingRegressor(loss='squared_error', max_depth=25, max_leaf_nodes=40)
)

pipeline_tuned.fit(X_train, y_train)
y_pred = pipeline_tuned.predict(X_test)

print("Tuned!")
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))

In [ ]:
pred = pd.DataFrame(y_pred).value_counts()
test = pd.DataFrame(y_test).value_counts()

print(pred.describe())
print(test.describe())